In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [13]:
books_df = pd.read_csv('../data/raw/books.csv')

In [14]:
print(f"First 5 rows of books data:\n{books_df.head()}")

First 5 rows of books data:
   Unnamed: 0  index                            authors  average_rating  \
0           0      0                ['Suzanne Collins']            4.34   
1           1      1  ['J.K. Rowling', 'Mary GrandPré']            4.44   
2           2      2                ['Stephenie Meyer']            3.57   
3           3      3                     ['Harper Lee']            4.25   
4           4      4            ['F. Scott Fitzgerald']            3.89   

   best_book_id  book_id  books_count  \
0       2767052        1          272   
1             3        2          491   
2         41865        3          226   
3          2657        4          487   
4          4671        5         1356   

                                         description  \
0  WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...   
1  Harry Potter's life is miserable. His parents ...   
2  About three things I was absolutely positive. ...   
3  The unforgettable novel of a childhood in a sl.

In [15]:
book_ratings_count = books_df['book_id'].value_counts()
print(f"Number of ratings per book:\n{book_ratings_count.head()}")

Number of ratings per book:
book_id
1       1
7801    1
7793    1
7794    1
7795    1
Name: count, dtype: int64


In [16]:
def cleanData(df, columns_to_keep, *args):
  clean_books_df = df[columns_to_keep].copy(deep=True)
  clean_books_df['group_text'] = clean_books_df[list(args)].astype(str).agg(" ".join, axis=1)
  clean_books_df = clean_books_df[["book_id", "group_text"]]
  return clean_books_df

clean_books_df = cleanData(books_df, ["book_id", "description", "genres", "authors"], "description", "genres", "authors")
clean_books_df = clean_books_df.sort_values('book_id').reset_index(drop=True)
clean_books_df.to_csv('../data/processed/clean_books_df.csv', index=False)
clean_books_df = pd.read_csv('../data/processed/clean_books_df.csv')

In [17]:
def preprocessText(text):
  lowered_text = text.lower()

  remove_punctuation = re.sub(r'[^\w\s]', '', lowered_text)
  remove_numbers = re.sub(r'[^a-zA-Z\s]', ' ', remove_punctuation)
  clean_text = re.sub(r'\s+', ' ', remove_numbers).strip()

  return clean_text

In [18]:
def build_vectors(clean_books_text):
  vectorizer = TfidfVectorizer(
     stop_words='english', # ignore random english words that don't add meaning
     min_df = 2, # ignore words that appear in less than 2 books
     max_df = 0.9, # ignores words that appear in 90%+ books
     ngram_range=(1,2) # use unigram and bigram
  )
  result =  vectorizer.fit_transform(clean_books_text) # contains rows: books; columns: words
  return vectorizer, result

In [19]:
clean_books_training_text = clean_books_df['group_text'].apply(preprocessText)
print("Processed Text: ")
print(clean_books_training_text.head())

Processed Text: 
0    winning means fame and fortunelosing means cer...
1    harry potters life is miserable his parents ar...
2    about three things i was absolutely positive f...
3    the unforgettable novel of a childhood in a sl...
4    alternate cover edition isbn isbn the great ga...
Name: group_text, dtype: object


In [20]:
vector, result = build_vectors(clean_books_training_text)


# First 20 features
features = vector.get_feature_names_out() # all the words features that were extracted from text
idf_values = vector.idf_ # idf score for each feature (higher idf = more unique)
vocab_items = list(vector.vocabulary_.items()) # displays: (word, column of the word) in TF-IDF matrix
print("test: ", result[0]) # format: (row index aka book, column aka word) TF-IDF value

print("First 20 features:", features[:20])
print("First 20 IDFs:", idf_values[:20])
print("First 20 vocabulary items:", vocab_items[:20])

print('\nTop 10 most unique words (Highest IDF):')

# Get words and their scores
word_idf_scores = zip(features, idf_values)

# Sort by IDF score (descending) to see the most unique terms
for word, score in sorted(word_idf_scores, key=lambda x: x[1], reverse=True)[:10]:
    print(f"{word}: {score:.4f}")

test:  <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 89 stored elements and shape (1, 101979)>
  Coords	Values
  (0, 98580)	0.08465444942868672
  (0, 57323)	0.12432131615118658
  (0, 29705)	0.08125863677019025
  (0, 12648)	0.07092817237385647
  (0, 21076)	0.12355659710275288
  (0, 42869)	0.17088358023441408
  (0, 35036)	0.24707449028001027
  (0, 6783)	0.07863951984108855
  (0, 76607)	0.09019463430959773
  (0, 67991)	0.10269786435291106
  (0, 48954)	0.056483477039466344
  (0, 62879)	0.06919530649394472
  (0, 2401)	0.061765980182881555
  (0, 51053)	0.06532623879376917
  (0, 61199)	0.07568824079153526
  (0, 65454)	0.12355659710275288
  (0, 80950)	0.09933084932951078
  (0, 11577)	0.22196032658827491
  (0, 87137)	0.08337261102702195
  (0, 23818)	0.2271985604464783
  (0, 38976)	0.08834699787233755
  (0, 19029)	0.07954984789861526
  (0, 47916)	0.07469713254917298
  (0, 52276)	0.06986358380922193
  (0, 33309)	0.09404834618609557
  :	:
  (0, 87461)	0.10227279718569757
  (0, 1564

In [21]:
liked_book_id = 68 # example index of a book

def bookRecommendation(liked_book_id):
  book_id_list = clean_books_df['book_id'].tolist() # convert book_id column into a list
  book_vectors = result
  liked_index = book_id_list.index(liked_book_id) #finds the row that the liked_book_id is in
  liked_vector = book_vectors[liked_index]

  similarities = cosine_similarity(liked_vector, book_vectors).flatten()
  similarities[liked_index] = -1  # sets the book itself to index -1 so it won't be recommended

  top_indices = similarities.argsort()[-5:][::-1] # return top 5 indices

  recommended_books = []

  for i in top_indices:
    recommended_books.append(book_id_list[i])

  print("Recommended book IDs:", recommended_books)

bookRecommendation(liked_book_id)



Recommended book IDs: [237, 2292, 7843, 5246, 5903]
